# Brainstorming and Focus Group Quantitative Experimentation 2.3: **Difficult people** under **action correction** only

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-04-26 09:22:04
Current date and time (UTC):   2026-04-26 12:22:04

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.3.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Control


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:
        person.import_fragment("./fragments/difficult_person.agent.fragment.json")
        print(person.minibio(extended=False))


Alan Merrick is a 48 year old Administrative Officer (Benefits and Records), British, currently living in Manchester, United Kingdom.
Anthony Russo is a 42 year old Journeyman Electrician / Senior Field Technician, American, currently living in Cleveland, Ohio, USA.
Anya Calder-Mori is a 45 year old Freelance Graphic Designer, Conceptual Artist and Cultural Critic, British, currently living in Camberwell, London, UK.
Barbara Jean Pratt is a 68 year old Retiree (former assembly line worker / part-time volunteer at church thrift shop), American, currently living in Small town near Toledo, Ohio, USA.
Colin Arthur Matthews is a 42 year old Operations Manager (Mid-level), British, currently living in Manchester, UK.
Colin Murray is a 52 year old Benefits and Housing Support Officer, British, currently living in Salford, Greater Manchester, UK.
Connor Walsh is a 28 year old Senior Customer Service Associate / Shift Lead (Retail Grocery Chain), American, currently living in Cleveland, Ohio, U

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    print(i)
    people_groups.append(people[i:i+4]
    )

len(people_groups)

0
4
8


3

In [10]:
people_groups

[[TinyPerson(name='Alan Merrick'),
  TinyPerson(name='Anthony Russo'),
  TinyPerson(name='Anya Calder-Mori'),
  TinyPerson(name='Barbara Jean Pratt')],
 [TinyPerson(name='Colin Arthur Matthews'),
  TinyPerson(name='Colin Murray'),
  TinyPerson(name='Connor Walsh'),
  TinyPerson(name='Darren McCall')],
 [TinyPerson(name='Dean Bartlett'),
  TinyPerson(name='Declan Blackwell'),
  TinyPerson(name='Edgar Milton Crane'),
  TinyPerson(name='Leonard Victor Hale')]]

In [11]:
# The experiment refers to customers

if experiment_runner.get_active_experiment() == "Control":
    for person in people:
        person.action_generator.enable_reasoning_step = False
        person.action_generator.enable_quality_checks = False

elif experiment_runner.get_active_experiment() == "Treatment":    
    for person in people:
       person.action_generator.enable_reasoning_step = False
       person.action_generator.enable_quality_checks = True
       person.action_generator.max_attempts = 2
       person.action_generator.enable_regeneration = True
       person.action_generator.quality_threshold = 5

## Proposals

In [12]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [13]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [14]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [15]:
agent_propositions_scores={}
environment_propositions_scores={}

In [16]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        #if experiment_runner.get_active_experiment() == "Treatment":
        #    interventions = \
        #        Intervention.create_for_each(people)\
        #            .set_functional_precondition(lambda target: target.actions_count >=7)\
        #            .set_textual_precondition(
        #                """
        #                AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
        #                The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
        #                That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
        #                Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!
#
        #                How to compute the steps gap:
        #                1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
        #                    This information can be found in the simulation trajectory.
        #                2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
        #                3. The proposition is true if, and only if, the difference D is **greater than** 5.
        #                """)\
        #            .set_effect(lambda target: target.think("""
        #                                                    I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
        #                                                    I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
        #                                                    of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
        #                                                    I need to think of something **entirely** new and different.
        #                                                    To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
        #                                                    others, and then, based on that, I'll think again about a new unique idea.
        #                                                    """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [17]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-26 09:22:56,400 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 09:22:56,422 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-26 09:23:00,040 - ThreadPoolExecutor-0_3(3916) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:23:00,132 - ThreadPoolExecutor-0_0(20856) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:23:01,159 - ThreadPoolExecutor-0_0(20856) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 09:23:01,164 - ThreadPoolExecutor-0_3(3916) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 09:23:01,350 - ThreadPoolExecutor-0_2(46972) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:23:01,428 - ThreadPoolExecutor-0_2(46972) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 09:23:01,506 - ThreadPoolExecutor-0_1(19684) - tin

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 09:24:52,874 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-26 09:24:55,214 - ThreadPoolExecutor-1_0(34072) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:24:55,222 - ThreadPoolExecutor-1_3(27284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:24:55,272 - ThreadPoolExecutor-1_0(34072) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 09:24:55,285 - ThreadPoolExecutor-1_3(27284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 09:24:55,341 - ThreadPoolExecutor-1_1(43980) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:24:55,390 - ThreadPoolExecutor-1_2(53236) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 09:24:55,398 - ThreadPoolExecutor-1_1(43980) - tinytroupe - INFO - Waiting

...

───────────────────────────────────────────── TinyWorld 4 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 10:16:49,058 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-26 10:16:51,061 - ThreadPoolExecutor-28_1(46912) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:16:51,092 - ThreadPoolExecutor-28_2(50304) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:16:51,111 - ThreadPoolExecutor-28_0(46708) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:16:51,118 - ThreadPoolExecutor-28_3(52160) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:16:51,148 - ThreadPoolExecutor-28_1(46912) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:16:51,170 - ThreadPoolExecutor-28_2(50304) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:16:51,186 - ThreadPoolExecutor-28_0(46708) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 4 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 10:17:37,868 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-26 10:17:40,269 - ThreadPoolExecutor-29_1(2588) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:17:40,284 - ThreadPoolExecutor-29_0(6684) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:17:40,343 - ThreadPoolExecutor-29_1(2588) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:17:40,352 - ThreadPoolExecutor-29_0(6684) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:17:40,474 - ThreadPoolExecutor-29_3(45324) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:17:40,511 - ThreadPoolExecutor-29_2(23260) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:17:40,544 - ThreadPoolExecutor-29_3(45324) - tinytroupe - INFO - Wait

({'Hard Persona Adherence': [3, 3, 2, 0, 3, 0, 2, 3, 2, 3, 1, 2, 3, 1, 2, 2],
  'Self-consistency': [9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 5, 9, 7, 9, 7, 9],
  'Fluency': [7, 7, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8, 8, 8, 8]},
 {'ideas_qty': [4, 4, 4, 3],
  'Task Completion': [9, 9, 9, 9],
  'Divergence': [3, 0, 0, 0]})

In [18]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-26 10:28:09,427 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 5 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 10:28:09,435 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-26 10:28:12,278 - ThreadPoolExecutor-32_2(45900) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:12,309 - ThreadPoolExecutor-32_3(1796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:12,362 - ThreadPoolExecutor-32_2(45900) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:28:12,419 - ThreadPoolExecutor-32_3(1796) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:28:12,504 - ThreadPoolExecutor-32_1(33492) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:12,602 - ThreadPoolExecutor-32_0(43112) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:12,656 - ThreadPoolExecutor-32_1(33492) - tinytroupe - INFO - Wa

───────────────────────────────────────────── TinyWorld 5 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 10:28:50,126 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-26 10:28:52,123 - ThreadPoolExecutor-33_2(42852) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:52,138 - ThreadPoolExecutor-33_1(51744) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:52,143 - ThreadPoolExecutor-33_0(34188) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:52,155 - ThreadPoolExecutor-33_3(47900) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 10:28:52,210 - ThreadPoolExecutor-33_2(42852) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:28:52,213 - ThreadPoolExecutor-33_1(51744) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 10:28:52,226 - ThreadPoolExecutor-33_3(47900) - tinytroupe - INFO - 

...

──────────────────────────────────────────── TinyWorld 10 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 11:47:11,666 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-26 11:47:13,682 - ThreadPoolExecutor-76_3(32164) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:47:13,700 - ThreadPoolExecutor-76_2(26976) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:47:13,721 - ThreadPoolExecutor-76_1(48952) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:47:13,728 - ThreadPoolExecutor-76_0(38992) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:47:13,753 - ThreadPoolExecutor-76_3(32164) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:47:13,806 - ThreadPoolExecutor-76_1(48952) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:47:13,815 - ThreadPoolExecutor-76_2(26976) - tinytroupe - INFO -

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 11:48:04,059 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-26 11:48:06,133 - ThreadPoolExecutor-77_1(39880) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:48:06,151 - ThreadPoolExecutor-77_2(648) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:48:06,172 - ThreadPoolExecutor-77_3(31596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:48:06,173 - ThreadPoolExecutor-77_0(50628) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:48:06,249 - ThreadPoolExecutor-77_1(39880) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:48:06,285 - ThreadPoolExecutor-77_2(648) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:48:06,294 - ThreadPoolExecutor-77_3(31596) - tinytroupe - INFO - Wai

({'Hard Persona Adherence': [3,
   3,
   2,
   0,
   3,
   0,
   2,
   3,
   2,
   3,
   1,
   2,
   3,
   1,
   2,
   2,
   1,
   2,
   6,
   2,
   0,
   2,
   0,
   4,
   1,
   0,
   2,
   3,
   0,
   3,
   3,
   0,
   2,
   2,
   0,
   3,
   0,
   3,
   1,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   5,
   9,
   7,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [7,
   7,
   8,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   8,
   8,
   7,
   7,
   8,
   9,
   6,
   9,
   9,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   7,
   8,
   9]},
 {'ideas_qty': [4, 4, 4, 3, 4, 4, 4, 3, 4, 4],
  'Task Completion': [9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
  'Divergence': [3, 0, 0, 0, 1, 0, 0, 1, 1, 0]})

In [19]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-26 11:58:00,825 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 11:58:00,834 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-26 11:58:02,758 - ThreadPoolExecutor-80_1(42460) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:02,771 - ThreadPoolExecutor-80_0(42012) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:02,771 - ThreadPoolExecutor-80_3(22580) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:02,787 - ThreadPoolExecutor-80_2(45008) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:02,854 - ThreadPoolExecutor-80_1(42460) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:58:02,862 - ThreadPoolExecutor-80_0(42012) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:58:02,882 - ThreadPoolExecutor-80_3(22580) - tinytroupe - INFO -

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 11:58:48,642 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-26 11:58:50,388 - ThreadPoolExecutor-81_1(36152) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:50,406 - ThreadPoolExecutor-81_0(36052) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:50,419 - ThreadPoolExecutor-81_3(14080) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:50,424 - ThreadPoolExecutor-81_2(10608) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 11:58:50,450 - ThreadPoolExecutor-81_1(36152) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:58:50,470 - ThreadPoolExecutor-81_0(36052) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 11:58:50,473 - ThreadPoolExecutor-81_3(14080) - tinytroupe - INFO -

...

──────────────────────────────────────────── TinyWorld 14 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 12:43:12,499 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-26 12:43:14,887 - ThreadPoolExecutor-108_3(42312) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:43:14,902 - ThreadPoolExecutor-108_0(48788) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:43:14,935 - ThreadPoolExecutor-108_1(7484) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:43:14,942 - ThreadPoolExecutor-108_2(18796) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:43:14,975 - ThreadPoolExecutor-108_0(48788) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:43:14,979 - ThreadPoolExecutor-108_3(42312) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:43:15,009 - ThreadPoolExecutor-108_1(7484) - tinytroupe - I

──────────────────────────────────────────── TinyWorld 14 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 12:44:00,109 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-26 12:44:02,122 - ThreadPoolExecutor-109_2(29320) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:44:02,170 - ThreadPoolExecutor-109_1(38128) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:44:02,177 - ThreadPoolExecutor-109_3(30524) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:44:02,177 - ThreadPoolExecutor-109_0(3748) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:44:02,217 - ThreadPoolExecutor-109_2(29320) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:44:02,247 - ThreadPoolExecutor-109_1(38128) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:44:02,263 - ThreadPoolExecutor-109_0(3748) - tinytroupe - I

({'Hard Persona Adherence': [3,
   3,
   2,
   0,
   3,
   0,
   2,
   3,
   2,
   3,
   1,
   2,
   3,
   1,
   2,
   2,
   1,
   2,
   6,
   2,
   0,
   2,
   0,
   4,
   1,
   0,
   2,
   3,
   0,
   3,
   3,
   0,
   2,
   2,
   0,
   3,
   0,
   3,
   1,
   0,
   0,
   0,
   3,
   2,
   0,
   0,
   0,
   0,
   2,
   6,
   2,
   4,
   0,
   5,
   0,
   2],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   5,
   9,
   7,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [7,
   7,
   8,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   8,
   8,
   7,
   7,
   8,
   9,
   6,
   9,
   9,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   8,
   7,
   8,
   9,
   8,
   7,
   9,
   8,
   8,

In [20]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-26 12:54:04,961 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 15 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 12:54:04,967 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-26 12:54:07,119 - ThreadPoolExecutor-112_2(33164) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:07,158 - ThreadPoolExecutor-112_0(37264) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:07,176 - ThreadPoolExecutor-112_2(33164) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:54:07,178 - ThreadPoolExecutor-112_1(26992) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:07,204 - ThreadPoolExecutor-112_0(37264) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:54:07,219 - ThreadPoolExecutor-112_3(40260) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:07,227 - ThreadPoolExecutor-112_1(26992) - tinytroupe -

──────────────────────────────────────────── TinyWorld 15 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 12:54:54,800 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-26 12:54:57,251 - ThreadPoolExecutor-113_0(40596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:57,257 - ThreadPoolExecutor-113_1(47476) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:57,258 - ThreadPoolExecutor-113_3(15436) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:57,303 - ThreadPoolExecutor-113_2(23780) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 12:54:57,329 - ThreadPoolExecutor-113_0(40596) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:54:57,347 - ThreadPoolExecutor-113_1(47476) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 12:54:57,354 - ThreadPoolExecutor-113_3(15436) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 20 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 14:02:44,600 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:02:46,381 - ThreadPoolExecutor-156_3(34048) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:02:46,403 - ThreadPoolExecutor-156_1(6968) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:02:46,457 - ThreadPoolExecutor-156_2(4732) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:02:46,471 - ThreadPoolExecutor-156_0(34496) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:02:46,501 - ThreadPoolExecutor-156_3(34048) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:02:46,520 - ThreadPoolExecutor-156_1(6968) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:02:46,568 - ThreadPoolExecutor-156_2(4732) - tinytroupe - INF

──────────────────────────────────────────── TinyWorld 20 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 14:03:31,681 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:03:33,413 - ThreadPoolExecutor-157_3(40600) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:03:33,417 - ThreadPoolExecutor-157_0(35212) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:03:33,429 - ThreadPoolExecutor-157_2(41920) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:03:33,454 - ThreadPoolExecutor-157_1(32912) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:03:33,476 - ThreadPoolExecutor-157_3(40600) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:03:33,481 - ThreadPoolExecutor-157_0(35212) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:03:33,493 - ThreadPoolExecutor-157_2(41920) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   2,
   0,
   3,
   0,
   2,
   3,
   2,
   3,
   1,
   2,
   3,
   1,
   2,
   2,
   1,
   2,
   6,
   2,
   0,
   2,
   0,
   4,
   1,
   0,
   2,
   3,
   0,
   3,
   3,
   0,
   2,
   2,
   0,
   3,
   0,
   3,
   1,
   0,
   0,
   0,
   3,
   2,
   0,
   0,
   0,
   0,
   2,
   6,
   2,
   4,
   0,
   5,
   0,
   2,
   0,
   1,
   2,
   3,
   1,
   2,
   0,
   0,
   3,
   3,
   0,
   1,
   1,
   4,
   1,
   3,
   1,
   0,
   2,
   3,
   0,
   2,
   1,
   2],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   5,
   9,
   7,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [21]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-26 14:12:26,643 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 14:12:26,651 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:12:29,158 - ThreadPoolExecutor-160_0(50244) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:12:29,205 - ThreadPoolExecutor-160_0(50244) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:12:29,354 - ThreadPoolExecutor-160_1(16612) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:12:29,360 - ThreadPoolExecutor-160_3(52948) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:12:29,382 - ThreadPoolExecutor-160_2(46488) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:12:29,408 - ThreadPoolExecutor-160_1(16612) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:12:29,420 - ThreadPoolExecutor-160_3(52948) - tinytroupe -

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 14:13:07,750 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:13:09,818 - ThreadPoolExecutor-161_3(27028) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:13:09,826 - ThreadPoolExecutor-161_0(36084) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:13:09,841 - ThreadPoolExecutor-161_1(49852) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:13:09,841 - ThreadPoolExecutor-161_2(49932) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:13:09,899 - ThreadPoolExecutor-161_3(27028) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:13:09,928 - ThreadPoolExecutor-161_1(49852) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:13:09,931 - ThreadPoolExecutor-161_0(36084) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 24 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 14:55:21,225 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:55:23,244 - ThreadPoolExecutor-188_1(27180) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:55:23,257 - ThreadPoolExecutor-188_2(49576) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:55:23,313 - ThreadPoolExecutor-188_0(43100) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:55:23,322 - ThreadPoolExecutor-188_1(27180) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:55:23,342 - ThreadPoolExecutor-188_2(49576) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:55:23,346 - ThreadPoolExecutor-188_3(36012) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:55:23,385 - ThreadPoolExecutor-188_0(43100) - tinytroupe -

──────────────────────────────────────────── TinyWorld 24 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 14:56:20,760 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-26 14:56:22,554 - ThreadPoolExecutor-189_0(36860) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:56:22,563 - ThreadPoolExecutor-189_1(22044) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:56:22,563 - ThreadPoolExecutor-189_2(16664) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:56:22,615 - ThreadPoolExecutor-189_3(35348) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 14:56:22,656 - ThreadPoolExecutor-189_0(36860) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:56:22,665 - ThreadPoolExecutor-189_1(22044) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 14:56:22,696 - ThreadPoolExecutor-189_2(16664) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   2,
   0,
   3,
   0,
   2,
   3,
   2,
   3,
   1,
   2,
   3,
   1,
   2,
   2,
   1,
   2,
   6,
   2,
   0,
   2,
   0,
   4,
   1,
   0,
   2,
   3,
   0,
   3,
   3,
   0,
   2,
   2,
   0,
   3,
   0,
   3,
   1,
   0,
   0,
   0,
   3,
   2,
   0,
   0,
   0,
   0,
   2,
   6,
   2,
   4,
   0,
   5,
   0,
   2,
   0,
   1,
   2,
   3,
   1,
   2,
   0,
   0,
   3,
   3,
   0,
   1,
   1,
   4,
   1,
   3,
   1,
   0,
   2,
   3,
   0,
   2,
   1,
   2,
   1,
   2,
   7,
   2,
   0,
   9,
   2,
   1,
   1,
   6,
   0,
   1,
   0,
   0,
   4,
   3],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   5,
   9,
   7,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [22]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-26 15:05:21,232 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 25 step 1 of 1 ─────────────────────────────────────────────

2026-04-26 15:05:21,240 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-26 15:05:23,544 - ThreadPoolExecutor-192_0(8116) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:05:23,600 - ThreadPoolExecutor-192_0(8116) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 15:05:23,635 - ThreadPoolExecutor-192_1(7860) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:05:23,659 - ThreadPoolExecutor-192_3(52068) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:05:23,682 - ThreadPoolExecutor-192_1(7860) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 15:05:23,710 - ThreadPoolExecutor-192_2(32892) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:05:23,727 - ThreadPoolExecutor-192_3(52068) - tinytroupe - INF

──────────────────────────────────────────── TinyWorld 25 step 1 of 5 ─────────────────────────────────────────────

2026-04-26 15:06:02,226 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-26 15:06:04,117 - ThreadPoolExecutor-193_0(47056) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:06:04,146 - ThreadPoolExecutor-193_3(52052) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:06:04,151 - ThreadPoolExecutor-193_1(52004) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:06:04,169 - ThreadPoolExecutor-193_2(35908) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 15:06:04,189 - ThreadPoolExecutor-193_0(47056) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 15:06:04,211 - ThreadPoolExecutor-193_3(52052) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 15:06:04,229 - ThreadPoolExecutor-193_1(52004) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 30 step 4 of 5 ─────────────────────────────────────────────

2026-04-26 16:15:25,147 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-26 16:15:27,423 - ThreadPoolExecutor-236_0(52672) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:15:27,437 - ThreadPoolExecutor-236_3(47088) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:15:27,442 - ThreadPoolExecutor-236_1(33304) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:15:27,442 - ThreadPoolExecutor-236_2(33408) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:15:27,513 - ThreadPoolExecutor-236_3(47088) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 16:15:27,518 - ThreadPoolExecutor-236_0(52672) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 16:15:27,524 - ThreadPoolExecutor-236_2(33408) - tinytroupe -

──────────────────────────────────────────── TinyWorld 30 step 5 of 5 ─────────────────────────────────────────────

2026-04-26 16:16:18,137 - MainThread(47004) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-26 16:16:19,961 - ThreadPoolExecutor-237_1(49688) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:16:19,995 - ThreadPoolExecutor-237_0(15012) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:16:20,013 - ThreadPoolExecutor-237_1(49688) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 16:16:20,055 - ThreadPoolExecutor-237_2(49696) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:16:20,063 - ThreadPoolExecutor-237_0(15012) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-26 16:16:20,073 - ThreadPoolExecutor-237_3(38788) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-26 16:16:20,118 - ThreadPoolExecutor-237_2(49696) - tinytroupe -

({'Hard Persona Adherence': [3,
   3,
   2,
   0,
   3,
   0,
   2,
   3,
   2,
   3,
   1,
   2,
   3,
   1,
   2,
   2,
   1,
   2,
   6,
   2,
   0,
   2,
   0,
   4,
   1,
   0,
   2,
   3,
   0,
   3,
   3,
   0,
   2,
   2,
   0,
   3,
   0,
   3,
   1,
   0,
   0,
   0,
   3,
   2,
   0,
   0,
   0,
   0,
   2,
   6,
   2,
   4,
   0,
   5,
   0,
   2,
   0,
   1,
   2,
   3,
   1,
   2,
   0,
   0,
   3,
   3,
   0,
   1,
   1,
   4,
   1,
   3,
   1,
   0,
   2,
   3,
   0,
   2,
   1,
   2,
   1,
   2,
   7,
   2,
   0,
   9,
   2,
   1,
   1,
   6,
   0,
   1,
   0,
   0,
   4,
   3,
   1,
   1,
   3,
   2,
   2,
   0,
   0,
   6,
   2,
   3,
   3,
   4,
   1,
   1,
   1,
   1,
   2,
   3,
   3,
   0,
   2,
   3,
   0,
   2],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   6,
   9,
   5,
   9,
   7,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,

In [23]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [24]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [26]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [27]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [3,
                0,
                0,
                0,
                1,
                0,
                0,
                1,
                1,
                0,
                1,
                3,
                2,
                2,
                0,
                0,
                2,
                2,
                1,
                1,
                0,
                0,
                0,
                0,
                0,
                0,
                0,
                0,
                0,
                0],
 'Fluency': [7,
             7,
             8,
             8,
             8,
             8,
             9,
             8,
             8,
             8,
             9,
             8,
             8,
             8,
             8,
             8,
             7,
             7,
             8,
             9,
             6,
             9,
             9,
             8,
             8,
             8,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,1.825000,1.683646,120.0
1,Self-consistency,8.816667,0.829937,120.0
2,Fluency,7.941667,1.204305,120.0
3,ideas_qty,3.785714,0.498675,28.0
4,Task Completion,8.766667,0.773854,30.0
5,Divergence,0.666667,0.958927,30.0


In [28]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [29]:
experiment_runner.finish_active_experiment()

2026-04-26 16:25:31,614 - MainThread(47004) - tinytroupe - INFO - Experiment 'Control' marked as finished.


True